In [4]:
#!/usr/bin/env python3

import json
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from catboost import (
    CatBoostClassifier,
    Pool,
)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
)

###############################################################################
# CONFIG
###############################################################################

DATA_ROOT = Path("../../data/v3/task2/agent_input")
ARTIFACT_DIR = Path("artifacts_task2")

TREATMENT_FILE = "prostate-treatment-decision.json"
REASONING_FILE = "prostate-treatment-decision-reasoning.json"
PROMPT_FILE = "structured-prompt.json"

N_SPLITS = 5
RANDOM_STATE = 42

FEATURES = [
    "bx_isup",
    "pirads",
    "psa",
    "age",
    "ct",
    "bx_gl_prim",
    "bx_gl_sec",
    "comorbidity",
    "psad",
]

CAT_COLS = [
    "bx_isup",
    "pirads",
    "ct",
]

###############################################################################
# LOAD
###############################################################################

rows = []

for case_dir in sorted(DATA_ROOT.iterdir()):

    if not case_dir.is_dir():
        continue

    treatment_file = case_dir / TREATMENT_FILE
    reasoning_file = case_dir / REASONING_FILE
    prompt_file = case_dir / PROMPT_FILE

    if not treatment_file.exists():
        continue

    if not reasoning_file.exists():
        continue

    if not prompt_file.exists():
        continue

    try:

        treatment = json.loads(
            treatment_file.read_text()
        )

        reasoning = json.loads(
            reasoning_file.read_text()
        )

        prompt = json.loads(
            prompt_file.read_text()
        )

    except Exception:
        continue

    #
    # collapse watchful waiting
    #

    if treatment == "watchful_waiting":
        treatment = "continued_surveillance"

    row = {
        "case_id": case_dir.name,
        "treatment": treatment,
        "confidence": reasoning.get(
            "confidence"
        ),
        "variable_weights": reasoning.get(
            "variable_weights",
            {},
        ),
    }

    for feature in FEATURES:
        row[feature] = prompt.get(feature)

    rows.append(row)

df = pd.DataFrame(rows)

print("Loaded:", len(df))

###############################################################################
# CLEAN
###############################################################################

for col in FEATURES:

    if col in CAT_COLS:

        df[col] = (
            df[col]
            .astype(str)
            .fillna("missing")
        )

    else:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )

        df[col] = df[col].fillna(
            df[col].median()
        )

###############################################################################
# DATA
###############################################################################

X = df[FEATURES]
y = df["treatment"]

cat_idx = [
    FEATURES.index(c)
    for c in CAT_COLS
]

###############################################################################
# FOLDS
###############################################################################

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_assignments = {}

oof_pred = np.empty(
    len(df),
    dtype=object,
)

oof_prob = np.zeros(
    (
        len(df),
        y.nunique(),
    )
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

###############################################################################
# BUILD PER-FOLD ARTIFACTS
###############################################################################

for fold_idx, (
    train_idx,
    test_idx,
) in enumerate(skf.split(X, y)):

    print()
    print(f"Fold {fold_idx}")

    train_df = df.iloc[
        train_idx
    ].copy()

    test_df = df.iloc[
        test_idx
    ].copy()

    #
    # fold assignments
    #

    for case_id in test_df["case_id"]:

        fold_assignments[
            case_id
        ] = fold_idx

    ###########################################################################
    # treatment model
    ###########################################################################

    model = CatBoostClassifier(
        iterations=300,
        depth=5,
        learning_rate=0.05,
        loss_function="MultiClass",
        random_seed=42,
        verbose=False,
    )

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx],
        cat_features=cat_idx,
    )

    model.save_model(
        ARTIFACT_DIR
        / f"treatment_model_fold_{fold_idx}.cbm"
    )

    pred = model.predict(
        X.iloc[test_idx]
    )

    prob = model.predict_proba(
        X.iloc[test_idx]
    )

    oof_pred[test_idx] = pred.flatten()
    oof_prob[test_idx] = prob

    ###########################################################################
    # weight priors
    ###########################################################################

    weight_priors = {}

    for treatment in sorted(
        train_df["treatment"].unique()
    ):

        weight_priors[
            treatment
        ] = {}

        for feature in FEATURES:

            counter = Counter()

            for vw in train_df[
                train_df["treatment"]
                == treatment
            ][
                "variable_weights"
            ]:

                if feature in vw:
                    counter[
                        vw[feature]
                    ] += 1

            total = sum(
                counter.values()
            )

            if total == 0:
                continue

            weight_priors[
                treatment
            ][feature] = {

                k: v / total

                for k, v in counter.items()
            }

    with open(
        ARTIFACT_DIR
        / f"weight_priors_fold_{fold_idx}.json",
        "w",
    ) as f:

        json.dump(
            weight_priors,
            f,
            indent=2,
        )

    ###########################################################################
    # confidence priors
    ###########################################################################

    confidence_priors = {}

    for treatment in sorted(
        train_df["treatment"].unique()
    ):

        subset = train_df[
            train_df["treatment"]
            == treatment
        ]

        counter = Counter(
            subset["confidence"]
        )

        total = sum(
            counter.values()
        )

        confidence_priors[
            treatment
        ] = {

            k: v / total

            for k, v in counter.items()
        }

    with open(
        ARTIFACT_DIR
        / f"confidence_priors_fold_{fold_idx}.json",
        "w",
    ) as f:

        json.dump(
            confidence_priors,
            f,
            indent=2,
        )

###############################################################################
# EVALUATOR CONSTANTS
###############################################################################

CONF_MAP = {
    "uncertain": 0,
    "borderline": 1,
    "clear": 2,
}

WEIGHT_MAP = {
    "not_used": 0,
    "noted": 1,
    "important": 2,
    "decisive": 3,
}

IMPORTANT_OR_DECISIVE = {
    "important",
    "decisive",
}

###############################################################################
# OOF CONFIDENCE + WEIGHTS
###############################################################################

oof_confidence = {}
oof_variable_weights = {}

for fold_idx, (
    train_idx,
    test_idx,
) in enumerate(skf.split(X, y)):

    train_df = df.iloc[train_idx]

    ###########################################################################
    # confidence priors
    ###########################################################################

    confidence_priors = {}

    for treatment in train_df["treatment"].unique():

        subset = train_df[
            train_df["treatment"] == treatment
        ]

        counter = Counter(
            subset["confidence"]
        )

        total = sum(
            counter.values()
        )

        confidence_priors[
            treatment
        ] = {

            k: v / total

            for k, v in counter.items()
        }

    ###########################################################################
    # weight priors
    ###########################################################################

    weight_priors = {}

    for treatment in train_df["treatment"].unique():

        weight_priors[treatment] = {}

        subset = train_df[
            train_df["treatment"] == treatment
        ]

        for feature in FEATURES:

            counter = Counter()

            for vw in subset[
                "variable_weights"
            ]:

                if feature in vw:

                    counter[
                        vw[feature]
                    ] += 1

            total = sum(
                counter.values()
            )

            if total == 0:
                continue

            weight_priors[
                treatment
            ][feature] = {

                k: v / total

                for k, v in counter.items()
            }

    ###########################################################################
    # OOF predictions
    ###########################################################################

    for row_idx in test_idx:

        pred_treatment = oof_pred[row_idx]

        #
        # confidence
        #

        conf_probs = confidence_priors.get(
            pred_treatment,
            {},
        )

        if conf_probs:

            pred_conf = max(
                conf_probs.items(),
                key=lambda x: x[1],
            )[0]

        else:

            pred_conf = "clear"

        oof_confidence[
            row_idx
        ] = pred_conf

        #
        # variable weights
        #

        pred_weights = {}

        for feature in FEATURES:

            probs = (
                weight_priors
                .get(pred_treatment, {})
                .get(feature, {})
            )

            if probs:

                pred_weights[
                    feature
                ] = max(
                    probs.items(),
                    key=lambda x: x[1],
                )[0]

            else:

                pred_weights[
                    feature
                ] = "not_used"

        oof_variable_weights[
            row_idx
        ] = pred_weights

###############################################################################
# OOF TREATMENT METRICS
###############################################################################

weighted_f1 = f1_score(
    y,
    oof_pred,
    average="weighted",
)

macro_f1 = f1_score(
    y,
    oof_pred,
    average="macro",
)

acc = accuracy_score(
    y,
    oof_pred,
)

print()
print("=== OOF TREATMENT PERFORMANCE ===")

print(
    "Macro F1:",
    round(macro_f1, 3),
)

print(
    "Weighted F1:",
    round(weighted_f1, 3),
)

print(
    "Accuracy:",
    round(acc, 3),
)

###############################################################################
# EVALUATOR-ALIGNED OOF METRICS
###############################################################################

conf_scores = []
vw_scores = []
factor_scores = []

for idx, row in df.iterrows():

    ###########################################################################
    # confidence score
    ###########################################################################

    gt_conf = row["confidence"]

    pred_conf = oof_confidence[idx]

    if (
        gt_conf in CONF_MAP
        and pred_conf in CONF_MAP
    ):

        dist = abs(
            CONF_MAP[gt_conf]
            - CONF_MAP[pred_conf]
        )

        conf_scores.append(
            1.0 - dist / 2.0
        )

    ###########################################################################
    # variable weight score
    ###########################################################################

    gt_weights = row[
        "variable_weights"
    ]

    pred_weights = (
        oof_variable_weights[idx]
    )

    errors = []

    for feature, gt_w in gt_weights.items():

        if gt_w not in WEIGHT_MAP:
            continue

        pred_w = pred_weights.get(
            feature,
            "not_used",
        )

        errors.append(
            abs(
                WEIGHT_MAP[gt_w]
                - WEIGHT_MAP[pred_w]
            ) / 3.0
        )

    if errors:

        vw_scores.append(
            1.0 - np.mean(errors)
        )

    ###########################################################################
    # important / decisive factor score
    ###########################################################################

    gt_set = {

        k

        for k, v in gt_weights.items()

        if v in IMPORTANT_OR_DECISIVE
    }

    pred_set = {

        k

        for k, v in pred_weights.items()

        if v in IMPORTANT_OR_DECISIVE
    }

    if not gt_set and not pred_set:

        factor_scores.append(
            1.0
        )

    elif not gt_set or not pred_set:

        factor_scores.append(
            0.0
        )

    else:

        tp = len(
            gt_set & pred_set
        )

        if tp == 0:

            factor_scores.append(
                0.0
            )

        else:

            precision = (
                tp
                / len(pred_set)
            )

            recall = (
                tp
                / len(gt_set)
            )

            factor_scores.append(
                2
                * precision
                * recall
                / (
                    precision
                    + recall
                )
            )

###############################################################################
# SUMMARY
###############################################################################

mean_confidence_score = np.mean(
    conf_scores
)

mean_variable_weight_score = np.mean(
    vw_scores
)

mean_factor_score = np.mean(
    factor_scores
)

#
# Evaluator weights when rationale score is absent
#

estimated_case_score = (
    0.225 * mean_confidence_score
    + 0.275 * mean_variable_weight_score
    + 0.175 * mean_factor_score
)

estimated_ranking_score = (
    weighted_f1
    + estimated_case_score
) / 2.0

print()
print(
    "=== OOF EVALUATOR METRICS ==="
)

print(
    "Confidence Score:",
    round(
        mean_confidence_score,
        3,
    )
)

print(
    "Variable Weight Score:",
    round(
        mean_variable_weight_score,
        3,
    )
)

print(
    "Important/Decisive F1:",
    round(
        mean_factor_score,
        3,
    )
)

print(
    "Estimated Case Score:",
    round(
        estimated_case_score,
        3,
    )
)

print(
    "Estimated Ranking Score:",
    round(
        estimated_ranking_score,
        3,
    )
)

###############################################################################
# SAVE OOF PREDICTIONS
###############################################################################

oof_rows = []

for idx, row in df.iterrows():

    oof_rows.append(
        {
            "case_id": row["case_id"],
            "fold": fold_assignments[
                row["case_id"]
            ],
            "gt_treatment": row["treatment"],
            "pred_treatment": oof_pred[
                idx
            ],
            "gt_confidence": row["confidence"],
            "pred_confidence": oof_confidence[
                idx
            ],
            "gt_variable_weights": row[
                "variable_weights"
            ],
            "pred_variable_weights": (
                oof_variable_weights[idx]
            ),
        }
    )

with open(
    ARTIFACT_DIR
    / "oof_predictions.json",
    "w",
) as f:

    json.dump(
        oof_rows,
        f,
        indent=2,
    )

###############################################################################
# SAVE FOLD MAP
###############################################################################

with open(
    ARTIFACT_DIR
    / "fold_assignments.json",
    "w",
) as f:

    json.dump(
        fold_assignments,
        f,
        indent=2,
    )

###############################################################################
# GLOBAL MODEL
###############################################################################

global_model = CatBoostClassifier(
    iterations=300,
    depth=5,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
)

global_model.fit(
    X,
    y,
    cat_features=cat_idx,
)

global_model.save_model(
    ARTIFACT_DIR
    / "treatment_model_all.cbm"
)

###############################################################################
# METADATA
###############################################################################

metadata = {
    "n_cases": len(df),
    "n_splits": N_SPLITS,
    "features": FEATURES,
    "cat_cols": CAT_COLS,
    "classes": sorted(
        df["treatment"].unique()
    ),
    "oof_macro_f1": float(
        macro_f1
    ),
    "oof_weighted_f1": float(
        weighted_f1
    ),
    "oof_confidence_score": float(
        mean_confidence_score
    ),
    "oof_variable_weight_score": float(
        mean_variable_weight_score
    ),
    "oof_factor_score": float(
        mean_factor_score
    ),
    "estimated_ranking_score": float(
        estimated_ranking_score
    ),
}

with open(
    ARTIFACT_DIR / "metadata.json",
    "w",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
    )

print()
print(
    "Artifacts written to:",
    ARTIFACT_DIR,
)


Loaded: 72

Fold 0

Fold 1

Fold 2

Fold 3

Fold 4

=== OOF TREATMENT PERFORMANCE ===
Macro F1: 0.849
Weighted F1: 0.848
Accuracy: 0.847

=== OOF EVALUATOR METRICS ===
Confidence Score: 0.903
Variable Weight Score: 0.807
Important/Decisive F1: 0.644
Estimated Case Score: 0.538
Estimated Ranking Score: 0.693

Artifacts written to: artifacts_task2
